# PV control parameters

Goal: inspect the PV ramp tests and decide what the EMS can safely assume from the panel.

For each distance test I calculate:

PV power = corrected_voltage * corrected_current

Then I compare the PV measurements to EMS needs:

1. supply the maximum scaled load, 100 mW
2. charge the battery at the EMS-scaled battery power
3. collect enough PV energy for one useful PEM output event

The important part is the feasibility check. A missing voltage threshold means the measured PV test
did not reach that requirement.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# This makes the notebook work both from the repo root and from its own folder.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "data").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not find the project root folder containing data/")
    PROJECT_ROOT = PROJECT_ROOT.parent

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 120)


## 1. Load all PV ramp tests


In [ ]:
pv_folder = PROJECT_ROOT / "data/PV_test"
pv_files = sorted(pv_folder.glob("PV_*cm_ramp.csv"))

print("PV files:")
for file in pv_files:
    print(" -", file.name)


## 2. Correct the PV sensor readings

The PV data is measured on INA3. I use the same simple calibration used in the original analysis:

corrected_voltage = ina3_bus_V - 0.180

corrected_current_A = ina3_current_mA / 1000 + 0.000138

corrected_power_mW = corrected_voltage * corrected_current_A * 1000


In [ ]:
def load_pv_file(file):
    df = pd.read_csv(file)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["time_s"] = (df["timestamp"] - df["timestamp"].iloc[0]).dt.total_seconds()
    df["pv_voltage_V"] = df["ina3_bus_V"] - 0.180
    df["pv_current_A"] = df["ina3_current_mA"] / 1000 + 0.000138
    df["pv_power_mW"] = df["pv_voltage_V"] * df["pv_current_A"] * 1000
    return df

pv_tests = {}
for file in pv_files:
    distance_cm = int(file.stem.split("_")[1].replace("cm", ""))
    pv_tests[distance_cm] = load_pv_file(file)

display(pv_tests[1].head())


## 3. Plot voltage, current, and power for each ramp


In [ ]:
plt.figure(figsize=(10, 4))
for distance_cm, df in pv_tests.items():
    plt.plot(df["time_s"], df["pv_power_mW"], label=f"{distance_cm} cm")
plt.title("PV power during load ramp")
plt.xlabel("Time [s]")
plt.ylabel("PV power [mW]")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(6, 5))
for distance_cm, df in pv_tests.items():
    df = df.sort_values("time_s").copy()

    jump = ((df["pv_voltage_V"].diff() ** 2) +
            ((df["pv_current_A"] * 1000).diff() ** 2)) ** 0.5

    df.loc[jump > 20, ["pv_voltage_V", "pv_current_A"]] = np.nan

    plt.plot(df["pv_voltage_V"], df["pv_current_A"] * 1000, label=f"{distance_cm} cm")

plt.title("PV current-voltage curves")
plt.xlabel("PV voltage [V]")
plt.ylabel("PV current [mA]")
plt.legend()
plt.grid(True)
plt.show()


## 4. Define the EMS power requirements


In [ ]:
# These values come from the sibling notebooks in control_parameters_new,
# not from the generated app/python/data/processed_* files.
LOAD_REQUIREMENT_MW = 100.00  # demand/demand_thresholds.ipynb: MAX_DEMAND_MILLIWATT
BATTERY_CHARGE_REQUIREMENT_MW = 100.00  # battery/battery_SOC.ipynb: BATTERY_MAX_DISCHARGE_POWER_W * 1000

# Derived in pem/hydrogen_discharge_rate.ipynb. This is an energy target, not a fixed PEM current.
USEFUL_PEM_OUTPUT_DURATION_S = 60.0
PEM_USEFUL_OUTPUT_ENERGY_J = 1.917
PEM_HYDROGEN_FOR_USEFUL_OUTPUT_ML = 0.911
PEM_CHARGE_ENERGY_FOR_USEFUL_OUTPUT_J = 12.82

requirements = pd.DataFrame(
    {
        "requirement": [
            "maximum EMS load",
            "battery charging",
            "PEM charge energy for one useful output event",
        ],
        "value": [
            LOAD_REQUIREMENT_MW,
            BATTERY_CHARGE_REQUIREMENT_MW,
            PEM_CHARGE_ENERGY_FOR_USEFUL_OUTPUT_J,
        ],
        "unit": ["mW", "mW", "J"],
    }
)
display(requirements)

## 5. Find MPP and first voltage that reaches each requirement

For a threshold voltage I use the first point in the ramp where PV power reaches the requirement.
This is intentionally conservative for the measured ramp order.


In [ ]:
def first_voltage_reaching_power(df, required_power_mW):
    reached = df[df["pv_power_mW"] >= required_power_mW]
    if reached.empty:
        return np.nan
    return reached.iloc[0]["pv_voltage_V"]


def pv_energy_trace(df):
    ramp = df.sort_values("time_s").copy()
    ramp["dt_s"] = ramp["time_s"].diff().fillna(0)
    # Negative corrected PV power is measurement noise or reverse flow, not useful PEM charge energy.
    ramp["positive_pv_power_W"] = ramp["pv_power_mW"].clip(lower=0) / 1000
    ramp["pv_energy_J"] = (ramp["positive_pv_power_W"] * ramp["dt_s"]).cumsum()
    return ramp


def first_point_reaching_energy(df, required_energy_J):
    ramp = pv_energy_trace(df)
    reached = ramp[ramp["pv_energy_J"] >= required_energy_J]
    if reached.empty:
        return pd.Series({"time_s": np.nan, "pv_voltage_V": np.nan, "pv_energy_J": ramp["pv_energy_J"].iloc[-1]})
    return reached.iloc[0][["time_s", "pv_voltage_V", "pv_energy_J"]]


rows = []
for distance_cm, df in pv_tests.items():
    mpp_idx = df["pv_power_mW"].idxmax()
    mpp = df.loc[mpp_idx]
    useful_charge_point = first_point_reaching_energy(df, PEM_CHARGE_ENERGY_FOR_USEFUL_OUTPUT_J)
    total_pv_energy_J = pv_energy_trace(df)["pv_energy_J"].iloc[-1]
    rows.append(
        {
            "distance_cm": distance_cm,
            "mpp_time_s": mpp["time_s"],
            "mpp_voltage_V": mpp["pv_voltage_V"],
            "mpp_current_A": mpp["pv_current_A"],
            "mpp_power_mW": mpp["pv_power_mW"],
            "max_voltage_V": df["pv_voltage_V"].max(),
            "max_current_A": df["pv_current_A"].max(),
            "first_voltage_for_load_V": first_voltage_reaching_power(df, LOAD_REQUIREMENT_MW),
            "first_voltage_for_battery_charge_V": first_voltage_reaching_power(df, BATTERY_CHARGE_REQUIREMENT_MW),
            "pv_energy_total_J": total_pv_energy_J,
            "time_to_useful_pem_charge_s": useful_charge_point["time_s"],
            "voltage_at_useful_pem_charge_V": useful_charge_point["pv_voltage_V"],
        }
    )

pv_summary = pd.DataFrame(rows).sort_values("distance_cm")
display(pv_summary)

## 6. Critical feasibility check


In [ ]:
PV_MAX_POWER_MW = pv_summary["mpp_power_mW"].max()
PV_MAX_POWER_W = PV_MAX_POWER_MW / 1000

PV_MIN_LOAD_SUPPLY_VOLTAGE = pv_summary["first_voltage_for_load_V"].dropna().max()
PV_MIN_BATTERY_CHARGE_VOLTAGE = (
    pv_summary["first_voltage_for_battery_charge_V"].dropna().max()
    if pv_summary["first_voltage_for_battery_charge_V"].notna().any()
    else np.nan
)

PV_CAN_PRODUCE_USEFUL_PEM_CHARGE = pv_summary["time_to_useful_pem_charge_s"].notna().any()
PV_MIN_TIME_TO_USEFUL_PEM_CHARGE_S = (
    pv_summary["time_to_useful_pem_charge_s"].dropna().min()
    if PV_CAN_PRODUCE_USEFUL_PEM_CHARGE
    else np.nan
)
PV_CONSERVATIVE_TIME_TO_USEFUL_PEM_CHARGE_S = (
    pv_summary["time_to_useful_pem_charge_s"].dropna().max()
    if PV_CAN_PRODUCE_USEFUL_PEM_CHARGE
    else np.nan
)

print(f"PV max measured power: {PV_MAX_POWER_MW:.1f} mW")
print(f"Load requirement:      {LOAD_REQUIREMENT_MW:.1f} mW")
print(f"Battery charge need:   {BATTERY_CHARGE_REQUIREMENT_MW:.1f} mW")
print(f"PEM useful charge energy target: {PEM_CHARGE_ENERGY_FOR_USEFUL_OUTPUT_J:.2f} J")

print("\nCan PV supply maximum EMS load?", PV_MAX_POWER_MW >= LOAD_REQUIREMENT_MW)
print("Can PV charge the battery at EMS-scaled battery power?", PV_MAX_POWER_MW >= BATTERY_CHARGE_REQUIREMENT_MW)
print("Can PV collect enough energy for one useful PEM output event?", PV_CAN_PRODUCE_USEFUL_PEM_CHARGE)
print(f"Fastest tested time to useful PEM charge: {PV_MIN_TIME_TO_USEFUL_PEM_CHARGE_S:.1f} s")
print(f"Conservative tested time to useful PEM charge: {PV_CONSERVATIVE_TIME_TO_USEFUL_PEM_CHARGE_S:.1f} s")

## 7. Values to use in the app


In [ ]:
pv_parameters = pd.DataFrame(
    {
        "parameter": [
            "PV_MAX_POWER_W",
            "PV_MIN_LOAD_SUPPLY_VOLTAGE",
            "PV_MIN_BATTERY_CHARGE_VOLTAGE",
            "PEM_CHARGE_ENERGY_FOR_USEFUL_OUTPUT_J",
            "PV_CAN_PRODUCE_USEFUL_PEM_CHARGE",
            "PV_MIN_TIME_TO_USEFUL_PEM_CHARGE_S",
            "PV_CONSERVATIVE_TIME_TO_USEFUL_PEM_CHARGE_S",
        ],
        "value": [
            PV_MAX_POWER_W,
            PV_MIN_LOAD_SUPPLY_VOLTAGE,
            PV_MIN_BATTERY_CHARGE_VOLTAGE,
            PEM_CHARGE_ENERGY_FOR_USEFUL_OUTPUT_J,
            PV_CAN_PRODUCE_USEFUL_PEM_CHARGE,
            PV_MIN_TIME_TO_USEFUL_PEM_CHARGE_S,
            PV_CONSERVATIVE_TIME_TO_USEFUL_PEM_CHARGE_S,
        ],
        "unit": ["W", "V", "V", "J", "bool", "s", "s"],
        "meaning": [
            "Maximum measured PV power",
            "Conservative voltage where measured PV can supply 100 mW load",
            "NaN means the measured PV did not reach battery charge power",
            "PEM input energy needed to later deliver one useful output event",
            "True if any measured PV test accumulated that much positive energy",
            "Fastest measured time to accumulate useful PEM charge energy",
            "Slowest measured successful time to accumulate useful PEM charge energy",
        ],
    }
)

display(pv_parameters)